In [1]:
# import modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
#from scipy.stats import vonmises_fisher

# load additional modules for lm fit unpacking and data loading
import pickle
import xarray as xr

# load the result object
from holopy.core.holopy_object import HoloPyObject, FullLoader
from holopy.core.utils import ensure_array, dict_without
import yaml
import importlib
# load using h5py
import h5py as h5

import holopy as hp
from holopy.core.process import normalize, bg_correct, center_find, subimage

# added this since seems not to be loading updated prior function correctly
#from holopy.core import prior

from holopy.scattering import Sphere, Spheres, calc_holo
from holopy.inference import prior, ExactModel, CmaStrategy, EmceeStrategy, AlphaModel, NmpfitStrategy
from holopy.inference import model

In [2]:
#needed to make display work properly (there are other options as well if this fails)
%matplotlib tk

In [3]:
# path to directory
DIRECTORYPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/'

## Troubleshooting lm unpacking

In [5]:

SPACING = 0.177
WAVELEN = 0.660
MEDIUM_INDEX = 1.33
POLARIZATION = [0.7, 0.7] # Calibrated 2020-09-02
path = '/Volumes/manoharan_lab/cmartin/Data/09-21-21/depletion/0.0875/03/'
MCMCSAVEPATH = '/Volumes/manoharan_lab/kaitorrens/Analysis/Fits/depletion_holography/09-21-21/depletion/0.0875/03/test/mcmc'
SAVEPATH = '/Volumes/manoharan_lab/kaitorrens/Analysis/Fits/depletion_holography/09-21-21/depletion/0.0875/03/test/lm_fits/'
PARAMSAVEPATH = '/Volumes/manoharan_lab/kaitorrens/Analysis/Fits/depletion_holography/09-21-21/depletion/0.0875/03/test/all_frames/'

def create_kaimodel(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    theta = parameters['theta']
    phi = parameters['phi']
    gap = parameters['gap']
    alpha = parameters['alpha']
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return model.KaiModel(scatterer, alpha=alpha)

def find_traj(mcmc_params):
    s1_r = mcmc_params['r_1']
    s2_r = mcmc_params['r_2']
    
    s1_n = mcmc_params['n_1']
    s2_n = mcmc_params['n_2']
    
    center_x = mcmc_params['x_g']
    center_y = mcmc_params['y_g']
    center_z = mcmc_params['z_g']
    gap = mcmc_params['gap']
    theta = mcmc_params['theta']
    phi = mcmc_params['phi']
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
    trajectory = [s1_center[0], s1_center[1], s1_center[2], 
                  s2_center[0], s2_center[1], s2_center[2]]
    return trajectory

    
# set range of fits you want to unpack
fits = np.arange(0, 10)

gap = np.zeros(len(fits))
gap_plus = np.zeros(len(fits))
gap_minus = np.zeros(len(fits))
z = np.zeros(len(fits))
phis = np.zeros(len(fits))
thetas = np.zeros(len(fits))
r1 = np.zeros(len(fits))
r2 = np.zeros(len(fits))
n1 = np.zeros(len(fits))
n2 = np.zeros(len(fits))

x1 = np.zeros(len(fits))
x2 = np.zeros(len(fits))
y1 = np.zeros(len(fits))
y2 = np.zeros(len(fits))
z1 = np.zeros(len(fits))
z2 = np.zeros(len(fits))

residuals = np.zeros(len(fits))


In [4]:
for i in range(len(fits)):

    # issue loading lm_fits using normal code so need to go into detail
    results_path = SAVEPATH + 'lm_dimer_fit_frame'+str(fits[i])+'.h5'

    attr_coords = '_attr_coords'
    def unpack_attrs(a):
        if len(a) == 0:
            return a
        new_attrs={}
        attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
        attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
        for attr in dict_without(attr_ref, attrs_to_ignore):
            if attr_ref[attr]:
                new_attrs[attr] = xr.DataArray(
                    a[attr],
                    coords=attr_ref[attr],
                    dims=list(attr_ref[attr].keys()))
            elif attr in a:
                if attr == 'noise_sd':
                    new_attrs[attr] = a[attr]
                else:
                    new_attrs[attr] = yaml.safe_load(a[attr])
            else:
                new_attrs[attr] = None
        return new_attrs

    with xr.open_dataset(results_path, engine='h5netcdf') as ds:
        if '_source_class' in ds.attrs:
            _source_class = ds.attrs.pop('_source_class')
            pathtok = _source_class.split('.')
            cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
            dataset = ds
            data = dataset.data
            data.attrs = unpack_attrs(data.attrs)
            if '_flat' in data.attrs.keys():
                flats = np.array(data.attrs['_flat']).T
                levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
                codes = [[level.index(f) for f in flat]
                        for level, flat in zip(levels, flats)]
                flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
                coordnames = list(data.coords)
                coordnames.remove('point')
                coords = {coord: data[coord] for coord in coordnames}
                coords['flat'] = flat_index
                data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                    coords=coords, attrs=data.attrs)
            # when loading mcmc fit need to comment out the next line -> seems like model is a problem because I have an sd attribute in the priors?
            print(dataset.attrs['model'])
            model = yaml.load(dataset.attrs['model'], Loader=FullLoader) #previously had this greyed out
            strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
            outlist = [data, model, strategy]
            #print(outlist)
            outlist.append(yaml.safe_load(dataset.attrs['time']))
            kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
            for key in ['lnprobs', 'samples', '_best_fit']:
                try:
                    kwargs[key] = getattr(dataset, key)
                    kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
                except AttributeError:
                    pass
            outlist.append(kwargs)
            # return args
            args = outlist
            return_variable = cls(*args)
    lm_fit = return_variable
    #lm_fit = hp.load(SAVEPATH + 'lm_dimer_fit_frame'+str(fits[i])+'.h5')
    dimer_holo = lm_fit.data
    params = lm_fit.parameters
    gap[i] = params['gap']
    gap_plus[i] = lm_fit.intervals[3].plus
    gap_minus[i] = lm_fit.intervals[3].minus
    z[i] = params['z_g']
    phis[i] = params['phi']
    thetas[i] = params['theta']
    r1[i] = params['r_1']
    r2[i] = params['r_2']
    n1[i] = params['n_1']
    n2[i] = params['n_2']
    
    trajectory = find_traj(params)
    x1[i] = trajectory[0]
    y1[i] = trajectory[1]
    z1[i] = trajectory[2]
    x2[i] = trajectory[3]
    y2[i] = trajectory[4]
    z2[i] = trajectory[5]

    # need to fix model loading with Phi and Theta priors to get the residuals to work
    #result = calc_holo(dimer_holo, lm_fit.scatterer, scaling=params['alpha'])
    #residuals[i] = np.sum((dimer_holo.values.flatten() - result.values.flatten())**2)
uncert_gap = {'gap':gap, 'gap_plus':gap_plus, 'gap_minus':gap_minus}
trajectory = {'x1':x1, 'y1':y1, 'z1':z1, 'x2':x2, 'y2':y2, 'z2':z2}
trajectory_df = pd.DataFrame(data=trajectory)

!KaiModel
_dummy_scatterer: !Spheres
  scatterers: [!Sphere {n: &id001 0, r: *id001, center: [*id001, *id001, *id001]},
    !Sphere {n: *id001, r: *id001, center: [*id001, *id001, *id001]}]
  warn: &id002 false
theory: !Multisphere
  niter: 200
  eps: 1.0e-06
  meth: &id004 1
  qeps1: 1.0e-05
  qeps2: 1.0e-08
  compute_escat_radial: *id002
  suppress_fortran_output: true
_parameters: [!BoundedGaussian {mu: 1.5841991961852777, sd: &id003 0.05, lower_bound: *id001,
    upper_bound: &id006 1.7, name: n_1}, !BoundedGaussian {mu: 0.6677633852095309,
    sd: *id003, lower_bound: *id001, upper_bound: &id007 1.0, name: r_1}, !BoundedGaussian {
    mu: 58.483363393561795, sd: &id005 0.177, lower_bound: 53.483363393561795, upper_bound: 63.483363393561795,
    name: x_g}, !BoundedGaussian {mu: 0.11531603641354699, sd: 0.005, lower_bound: *id001,
    upper_bound: 1.3, name: gap}, !Phi {mu: 0.05151714728026115, name: phi, sd: *id004},
  !Theta {mu: 2.651991762044066, name: theta, sd: *id004}, !Boun

TypeError: __init__() got an unexpected keyword argument 'sd'

In [6]:
print(lm_fit)

FitResult(data=<xarray.DataArray 'data' (z: 1, x: 200, y: 200)>
[40000 values with dtype=float64]
Coordinates:
  * z        (z) int64 0
  * x        (x) float64 40.71 40.89 41.06 41.24 ... 75.4 75.58 75.76 75.93
  * y        (y) float64 90.98 91.16 91.33 91.51 ... 125.7 125.8 126.0 126.2
Attributes:
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
    noise_sd:            0.00862557822556247, model=<module 'holopy.inference.model' from '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holopy_code/holopy/holopy/inference/model.py'>, strategy=NmpfitStrategy(quiet=True, ftol=1e-10, xtol=1e-10, gtol=1e-10, damp=0, maxiter=100), time=103.20202589035034)


# Parameters and old data loading code

In [4]:
# taken from Caroline's code
SPACING = 0.177
WAVELEN = 0.660
MEDIUM_INDEX = 1.33
POLARIZATION = POLARIZATION = [0.56, 0.83] # Calibrated 2020-09-02

#Sphere 1
R_1_MEAN = 0.6749016953839639
R_1_SIGMA =  0.00047233977835876834
N_1_MEAN =  1.5848484802283918
N_1_SIGMA =  0.00027320846407879903
#Sphere 2
R_2_MEAN =  0.6446649533295826
R_2_SIGMA =  0.000617682113114549
N_2_MEAN =  1.601784237444771
N_2_SIGMA =  0.0003353994780766957

DIMER_Z_GUESS = 20.0

## Working on loading data

In [4]:
SAVEPATH = DIRECTORYPATH + 'real_data_full_process_0.0875/03/'
print(SAVEPATH)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/real_data_full_process_0.0875/03/


In [99]:
# path that determines what fit you load
results_path = SAVEPATH+'dimer_frame'+str(frames[i])+'_cma4.h5'

In [5]:
results_path = SAVEPATH + 'dimer_frame0_nogap_new_priors_fix_loading_1.h5'

In [7]:
# fixed the model problem, but still have the numpy float problem
quick_loaded_fit = hp.load(results_path)
print(quick_loaded_fit)

FitResult(data=<xarray.DataArray 'data' (z: 1, x: 200, y: 200)>
array([[[1.01175653, 1.00863477, 1.01202496, ..., 1.02107954,
         1.02035833, 1.00273287],
        [1.01036751, 0.99406793, 0.9830042 , ..., 1.00622908,
         1.02170444, 1.02781277],
        [1.00020314, 0.998468  , 0.98871473, ..., 0.99738392,
         1.00447002, 1.01476335],
        ...,
        [1.01235947, 0.99355925, 1.00212442, ..., 0.98018861,
         0.97230096, 0.98421285],
        [1.02359349, 1.00426822, 0.99453393, ..., 0.98138699,
         0.99425139, 1.01041335],
        [1.01250025, 1.02688103, 0.99595373, ..., 0.99079656,
         0.9962202 , 1.00964035]]])
Coordinates:
  * z        (z) int64 0
  * x        (x) float64 40.71 40.89 41.06 41.24 ... 75.4 75.58 75.76 75.93
  * y        (y) float64 90.98 91.16 91.33 91.51 ... 125.7 125.8 126.0 126.2
Attributes:
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
 

In [8]:
# path for Caroline's fit
Caroline_load_path = '/Volumes/manoharan_lab/cmartin/Fits/depletion_holography/09-21-21/depletion/0.0875/03/mcmcdimer_frame0_cma.h5'
results_path = Caroline_load_path

In [20]:
# path for my fits on the groupshare
group_share_path = '/Volumes/manoharan_lab/kaitorrens/Analysis/Fits/depletion_holography/09-21-21/depletion/0.0875/03/test/mcmc'
results_path = group_share_path +'dimer_frame'+str(0)+'_mcmc.h5'

In [32]:
# problem with noise_sd where it is a np.float and not a string as the other coord values are
# and so yaml.safe_load in unpack_attrs(a[attr]) gets confused, fixed by putting in a
# specific if clause to catch noise_sd which is clunky but workable
attr_coords = '_attr_coords'
def unpack_attrs(a):
    if len(a) == 0:
        return a
    new_attrs={}
    attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
    #
    #print(attr_ref)
    attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
    for attr in dict_without(attr_ref, attrs_to_ignore):
        #print(attr)
        if attr_ref[attr]:
            new_attrs[attr] = xr.DataArray(
                a[attr],
                coords=attr_ref[attr],
                dims=list(attr_ref[attr].keys()))
            print("used first clause")
        elif attr in a:
            print("attempted second clause")
            print(isinstance(a[attr],np.float64))
            print(type(a[attr]))
            if isinstance(a[attr],np.float64):
                new_attrs[attr] = float(a[attr])
                print("and succeeded second clause using noise_sd specific clause")
            else:
                new_attrs[attr] = yaml.safe_load(a[attr])
                print("and succeeded second clause normally")
        else:
            new_attrs[attr] = None
    return new_attrs

with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    if '_source_class' in ds.attrs:
        _source_class = ds.attrs.pop('_source_class')
        pathtok = _source_class.split('.')
        cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
        #ds.close()
        #return_variable = cls._load(results_path)
    #def _load(cls, ds, **kwargs):
        #with xr.open_dataset(ds, engine='h5netcdf', **kwargs) as ds:
        #print(ds.load())
        dataset = ds
        data = dataset.data
        #
        #print(data.attrs)
        data.attrs = unpack_attrs(data.attrs)
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
            #print(data)
        print(dataset.attrs['model'])
        model = yaml.load(dataset.attrs['model'], Loader=FullLoader) #previously had this greyed out
        print(model)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        #print(outlist)
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        # return args
        args = outlist
        
        #args = cls._unserialize(ds.load())
        return_variable = cls(*args)

used first clause
attempted second clause
False
<class 'str'>
and succeeded second clause normally
attempted second clause
False
<class 'str'>
and succeeded second clause normally
attempted second clause
True
<class 'numpy.float64'>
and succeeded second clause using noise_sd specific clause
!KaiModel
_dummy_scatterer: !Spheres
  scatterers: [!Sphere {n: &id001 0, r: *id001, center: [*id001, *id001, *id001]},
    !Sphere {n: *id001, r: *id001, center: [*id001, *id001, *id001]}]
  warn: &id002 false
theory: !Multisphere
  niter: 200
  eps: 1.0e-06
  meth: 1
  qeps1: 1.0e-05
  qeps2: 1.0e-08
  compute_escat_radial: *id002
  suppress_fortran_output: true
_parameters: [!Phi {k: *id001, mu: 3.141592653589793, name: phi}, !Theta {k: *id001,
    mu: 1.5707963267948966, name: theta}, !Uniform {lower_bound: 5, upper_bound: 50,
    guess: 20.0, name: Z}, !Uniform {lower_bound: 0.5, upper_bound: 1.2, guess: 0.8,
    name: Alpha}]
_parameter_names: [phi, theta, Z, Alpha]
_maps: {model: [&id003 !!py

In [10]:
print(return_variable)

SamplingResult(data=<xarray.DataArray (flat: 8000)>
array([0.93718599, 1.02949085, 1.1157527 , ..., 0.99884039, 1.0255741 ,
       0.95243352])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 52.21 65.14 55.4 58.41 ... 68.85 43.54 69.21 70.09
  * y        (flat) float64 108.1 96.46 105.7 125.7 ... 97.53 108.3 94.52 116.6
  * z        (flat) int64 0 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0 0
Attributes:
    _flat:               [[52.214999999999996, 108.14699999999999, 0], [65.13...
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
    noise_sd:            0.00862557822556247
    original_dims:       {'x': [40.71, 40.887, 41.064, 41.241, 41.418, 41.595..., model=<module 'holopy.inference.model' from '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holopy_code/holopy/holopy/inference/model.py'>, strategy=EmceeStrategy(nwalkers=50, nsamples=1000, npix

Caroline's fit which loads okay: {'_attr_coords': '{illum_polarization: {vector: [x, y, z]}, illum_wavelen: &id001 false, medium_index: *id001,\n noise_sd: *id001}\n', 'illum_polarization': array([0.55930131, 0.82896444, 0. ]), 'illum_wavelen': '0.66\n...\n', 'medium_index': '1.33\n...\n', 'name': 'data', 'noise_sd': '- 0.00862557822556247\n'}

My fit which doesn't load: {'_attr_coords': '{illum_polarization: {vector: [x, y, z]}, illum_wavelen: &id001 false, medium_index: *id001,\n noise_sd: {}}\n', 'name': 'data', 'medium_index': '1.33\n...\n', 'illum_wavelen': '0.66\n...\n', 'illum_polarization': array([0.55930131, 0.82896444, 0. ]), 'noise_sd': 0.00862557822556247}

In [ ]:
print(dimer_holo)
print(dimer_holo.noise_sd)
print(dimer_holo.medium_index)

In [ ]:
print(img.raw)
print(img.processed)

In [ ]:
print(img.bg)
print(img.dc)

In [11]:
loaded_results = return_variable

In [102]:
# print result object properties
best_fit_values = loaded_results4.parameters
initial_guess_values = loaded_results4.guess_parameters
best_fit_sphere = loaded_results4.scatterer
best_fit_hologram = loaded_results4.hologram
best_fit_lnprob = loaded_results4.max_lnprob
print(loaded_results4.parameters)
print(loaded_results4.guess_parameters)
print(loaded_results4.max_lnprob)

{'Gap': 0.011653428064409298, 'Z': 21.078224784687794, 'Alpha': 0.6153847702962723}
{'Gap': 0.03, 'Z': 21.082428827213196, 'Alpha': 0.6167584786983463}
42219.476766027656


In [12]:
print(loaded_results)

SamplingResult(data=<xarray.DataArray (flat: 8000)>
array([0.93718599, 1.02949085, 1.1157527 , ..., 0.99884039, 1.0255741 ,
       0.95243352])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 52.21 65.14 55.4 58.41 ... 68.85 43.54 69.21 70.09
  * y        (flat) float64 108.1 96.46 105.7 125.7 ... 97.53 108.3 94.52 116.6
  * z        (flat) int64 0 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0 0
Attributes:
    _flat:               [[52.214999999999996, 108.14699999999999, 0], [65.13...
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
    noise_sd:            0.00862557822556247
    original_dims:       {'x': [40.71, 40.887, 41.064, 41.241, 41.418, 41.595..., model=<module 'holopy.inference.model' from '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holopy_code/holopy/holopy/inference/model.py'>, strategy=EmceeStrategy(nwalkers=50, nsamples=1000, npix

In [82]:
print(results3)

FitResult(data=<xarray.DataArray 'data' (z: 1, x: 200, y: 200)>
array([[[1.01175653, 1.00863477, 1.01202496, ..., 1.02107954,
         1.02035833, 1.00273287],
        [1.01036751, 0.99406793, 0.9830042 , ..., 1.00622908,
         1.02170444, 1.02781277],
        [1.00020314, 0.998468  , 0.98871473, ..., 0.99738392,
         1.00447002, 1.01476335],
        ...,
        [1.01235947, 0.99355925, 1.00212442, ..., 0.98018861,
         0.97230096, 0.98421285],
        [1.02359349, 1.00426822, 0.99453393, ..., 0.98138699,
         0.99425139, 1.01041335],
        [1.01250025, 1.02688103, 0.99595373, ..., 0.99079656,
         0.9962202 , 1.00964035]]])
Coordinates:
  * z        (z) int64 0
  * x        (x) float64 40.71 40.89 41.06 41.24 ... 75.4 75.58 75.76 75.93
  * y        (y) float64 90.98 91.16 91.33 91.51 ... 125.7 125.8 126.0 126.2
Attributes:
    medium_index:        1.33
    illum_wavelen:       0.66
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
 

In [97]:
print(Caroline_loaded)

FitResult(data=<xarray.DataArray 'data' (z: 1, x: 200, y: 200)>
array([[[1.01175653, 1.00863477, 1.01202496, ..., 1.02107954,
         1.02035833, 1.00273287],
        [1.01036751, 0.99406793, 0.9830042 , ..., 1.00622908,
         1.02170444, 1.02781277],
        [1.00020314, 0.998468  , 0.98871473, ..., 0.99738392,
         1.00447002, 1.01476335],
        ...,
        [1.01235947, 0.99355925, 1.00212442, ..., 0.98018861,
         0.97230096, 0.98421285],
        [1.02359349, 1.00426822, 0.99453393, ..., 0.98138699,
         0.99425139, 1.01041335],
        [1.01250025, 1.02688103, 0.99595373, ..., 0.99079656,
         0.9962202 , 1.00964035]]])
Coordinates:
  * x        (x) float64 40.71 40.89 41.06 41.24 ... 75.4 75.58 75.76 75.93
  * y        (y) float64 90.98 91.16 91.33 91.51 ... 125.7 125.8 126.0 126.2
  * z        (z) int64 0
Attributes:
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
 

In [98]:
initial_guess_values = Caroline_loaded.guess_parameters

In [31]:
#results3 = loaded_results3

In [29]:
# code used to debug the unpack function for loading cma fits
a = data.attrs
new_attrs={}
attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
#
print(attr_ref)
attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
for attr in dict_without(attr_ref, attrs_to_ignore):
    print(attr)
    print(attr_ref[attr])
    if attr_ref[attr]:
        new_attrs[attr] = xr.DataArray(
            a[attr],
            coords=attr_ref[attr],
            dims=list(attr_ref[attr].keys()))
        print("used first clause")
    elif attr in a:
        if attr == 'noise_sd':
            new_attrs[attr] = a[attr]
            print("used new if subclause in second clause")
        else:
            print(a[attr])
            print(type(a[attr]))
            print("attempted second clause")
            new_attrs[attr] = yaml.safe_load(a[attr])
            print(new_attrs[attr])
            print(type(new_attrs[attr]))
            print("and succeeded second clause")
    else:
        new_attrs[attr] = None
print(new_attrs)

{'_flat': [[52.214999999999996, 108.14699999999999, 0], [65.136, 96.46499999999999, 0], [55.400999999999996, 105.669, 0], [58.41, 125.66999999999999, 0], [58.940999999999995, 102.83699999999999, 0], [42.303, 97.881, 0], [61.596, 121.776, 0], [70.446, 126.201, 0], [51.33, 109.74, 0], [49.559999999999995, 103.19099999999999, 0], [67.25999999999999, 102.83699999999999, 0], [41.949, 124.96199999999999, 0], [58.233, 126.201, 0], [71.154, 115.93499999999999, 0], [53.631, 119.475, 0], [50.445, 105.84599999999999, 0], [54.339, 108.678, 0], [50.622, 91.155, 0], [42.657, 105.315, 0], [72.393, 105.669, 0], [44.604, 92.74799999999999, 0], [49.736999999999995, 91.68599999999999, 0], [61.065, 116.28899999999999, 0], [49.559999999999995, 95.58, 0], [44.25, 104.961, 0], [45.135, 94.695, 0], [50.622, 118.413, 0], [50.976, 120.36, 0], [47.613, 95.40299999999999, 0], [43.719, 120.53699999999999, 0], [67.083, 121.59899999999999, 0], [45.489, 117.705, 0], [42.833999999999996, 100.89, 0], [67.437, 107.97, 0

KeyError: '_attr_coords'

In [57]:
# list out model3._lnprior(pars) to see where things are going wrong and try to fix bug
if 'scatterer' in model3._maps:
    try:
        par_scat = model3._scatterer_from_parameters(results3._parameters)
    except InvalidScatterer:
        print( -np.inf)

for constraint in model3.constraints:
    if not constraint.check(par_scat):
        print( -np.inf)
            
sum_of_lnprob = 0
# changed to [] from np.nan since np.nan != np.nan is true and so was triggering undesired behavior
new_theta = []
new_phi = []
# currently requires theta and phi to be input as gaussian or bounded gaussian priors
# even though we then treat them as a joint prior von Mises–Fisher distribution

# loop through parameter values and corresponding priors
for p, val in zip(model3._parameters, results3._parameters):
    # if the prior is not for phi or theta
    if p.name != "theta" and p.name != "phi":
        print(sum_of_lnprob)
        sum_of_lnprob = p.lnprob(val) + sum_of_lnprob
    # if the prior is theta
    elif p.name == "theta":
        previous_theta = p.mu
        k_param = p.concentration_parameter
        new_theta = val
    # if the prior is phi
    elif p.name == "phi":
        previous_phi = p.mu
        k_param = p.concentration_parameter
        new_phi = val
print("finished looping through")
# if phi and theta are both parameters then use von Mises-Fisher joint distribution
if new_phi != [] and new_theta != []:
    print("triggered theta and phi loop")
    # k is concentration parameter k=0 is uniform distribution, k>0 is unimodal around average direction (set by p.mu)
    # add special case for k is 0 (need to check this is correct since we want uniform in surface of sphere not in angle)
    # I think this is correct for lnprob, but sampling so uniform would need to be different 
    if k_param == 0:
        # log of sin(theta) which compensates for overrepresentation at the poles combined with 1/surface area of unit sphere 
        # special case for theta = n*pi with n an integer to avoid -np.inf in likelihood (see uniform prior for another example of this)
        if np.sin(new_theta) == 0:
            ln_von_mises_fisher = -1/EPS
        else:
            ln_von_mises_fisher = np.log(np.sin(new_theta)/(4*np.pi))
    # if k is not zero
    else:
        # take dot product between past phi and theta and proposed phi and theta
        dot_product = (np.cos(new_phi-previous_phi)*np.sin(new_theta)*np.sin(previous_theta)
                    + np.cos(new_theta)*np.cos(previous_theta))
        # log of normalization of von Mises-Fisher in 3 dimensions
        ln_normal = np.log(k_param)-np.log(2*np.pi*(np.exp(k_param)-np.exp(-k_param)))
        # add log of normalization and dot product times k parameter
        ln_von_mises_fisher = ln_normal + k_param*dot_product
    sum_of_lnprob = ln_von_mises_fisher + sum_of_lnprob
print(sum_of_lnprob)

0
0.39318823518382906
-3.295691218930107
finished looping through
-2.9390162749913746


## Load fit if necessary

In [7]:
# path that determines what fit you load
results_path = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/normal_side_by_side_geometry/variable_initial_conditions_version_1/longer_fits/von_Mises_Fisher_nsample_2000_fit_1_mcmc.h5'

In [10]:
INITIALCONDPATH = GEOMETRYPATH + 'no_mod_initial_conditions_version_1/particle_swap_not_degenerate/'

In [11]:
# alternative way of getting results path (less explicit)
LOAD_NAME_OF_FIT = 'von_Mises_Fisher_walkers_50_nsample_1000_fit_1'
results_path = INITIALCONDPATH + LOAD_NAME_OF_FIT + '_mcmc.h5'

In [62]:
# Load one of Caroline's fits
Caroline_load_path = '/Volumes/manoharan_lab/cmartin/Fits/depletion_holography/09-21-21/depletion/0.0875/mcmcdimer_frame0_mcmc.h5'
Caroline_fit = hp.load(Caroline_load_path)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/compare_Caroline_fit/depletion_holography_09-21-21_depletion_0.0875_mcmcdimer_frame0


In [63]:
SAVEPATH = DIRECTORYPATH+ 'compare_Caroline_fit/depletion_holography_09-21-21_depletion_0.0875_mcmcdimer_frame0'
print(SAVEPATH)
results5 = Caroline_fit

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/compare_Caroline_fit/depletion_holography_09-21-21_depletion_0.0875_mcmcdimer_frame0


In [16]:
# try to load fit result object using hp.load code
# may need to modify code now that noise_sd is assigned as an attribute instead of a coord
attr_coords = '_attr_coords'
def unpack_attrs(a):
    if len(a) == 0:
        return a
    new_attrs={}
    attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
    attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
    for attr in dict_without(attr_ref, attrs_to_ignore):
        if attr_ref[attr]:
            new_attrs[attr] = xr.DataArray(
                a[attr],
                coords=attr_ref[attr],
                dims=list(attr_ref[attr].keys()))
        elif attr in a:
            new_attrs[attr] = yaml.safe_load(a[attr])
        else:
            new_attrs[attr] = None
    return new_attrs

with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    if '_source_class' in ds.attrs:
        _source_class = ds.attrs.pop('_source_class')
        pathtok = _source_class.split('.')
        cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
        #ds.close()
        #return_variable = cls._load(results_path)
    #def _load(cls, ds, **kwargs):
        #with xr.open_dataset(ds, engine='h5netcdf', **kwargs) as ds:
        print(ds.load())
        dataset = ds
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        # Kai added this to move noise_sd to correct place
        data.attrs['noise_sd'] = data.coords['noise_sd'].to_numpy()
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            # seems like noise_sd should be attribute not coordinate
            coordnames.remove('noise_sd') # added this
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
            print(data)
        print(dataset.attrs['model'])
        # seems like model is a problem because I have an sd attribute in the priors?
        #model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        # return args
        args = outlist
        
        #args = cls._unserialize(ds.load())
        return_variable = cls(*args)
'''
def _unserialize(cls, dataset):
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
        model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])

        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        return outlist
'''

<xarray.Dataset>
Dimensions:    (point: 8000, walker: 50, chain: 1000, parameter: 11)
Coordinates:
  * point      (point) int64 0 1 2 3 4 5 6 ... 7994 7995 7996 7997 7998 7999
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Dimensions without coordinates: walker, chain
Data variables:
    data       (point) float64 0.909 1.208 1.037 1.025 ... 1.007 0.933 1.019
    lnprobs    (walker, chain) float64 -7.421e+04 -7.421e+04 ... 2.67e+04
    samples    (walker, chain, parameter) float64 1.585 0.675 ... 0.6448 0.9994
Attributes:
    model:     !KaiModel\n_dummy_scatterer: !Spheres\n  scatterers: [!Sphere ...
    strategy:  !EmceeStrategy\nnwalkers: 50\nnsamples: 1000\nnpixels: 8000\nw...
    time:      2127.895318031311
    _kwargs:   {}\n


KeyError: 'noise_sd'

In [12]:
# modified since noise_sd is assigned as an attribute instead of a coord
attr_coords = '_attr_coords'
def unpack_attrs(a):
    if len(a) == 0:
        return a
    new_attrs={}
    attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
    attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
    for attr in dict_without(attr_ref, attrs_to_ignore):
        if attr_ref[attr]:
            new_attrs[attr] = xr.DataArray(
                a[attr],
                coords=attr_ref[attr],
                dims=list(attr_ref[attr].keys()))
        elif attr in a:
            new_attrs[attr] = yaml.safe_load(a[attr])
        else:
            new_attrs[attr] = None
    return new_attrs

with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    if '_source_class' in ds.attrs:
        _source_class = ds.attrs.pop('_source_class')
        pathtok = _source_class.split('.')
        cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
        #ds.close()
        #return_variable = cls._load(results_path)
    #def _load(cls, ds, **kwargs):
        #with xr.open_dataset(ds, engine='h5netcdf', **kwargs) as ds:
        print(ds.load())
        dataset = ds
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
            print(data)
        print(dataset.attrs['model'])
        # seems like model is a problem because I have an sd attribute in the priors?
        #model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        # return args
        args = outlist
        
        #args = cls._unserialize(ds.load())
        return_variable = cls(*args)

<xarray.Dataset>
Dimensions:    (point: 8000, walker: 50, chain: 1000, parameter: 11)
Coordinates:
  * point      (point) int64 0 1 2 3 4 5 6 ... 7994 7995 7996 7997 7998 7999
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Dimensions without coordinates: walker, chain
Data variables:
    data       (point) float64 0.9506 0.9849 1.008 1.027 ... 0.917 1.023 1.041
    lnprobs    (walker, chain) float64 1.431e+04 1.431e+04 ... 2.671e+04
    samples    (walker, chain, parameter) float64 1.585 0.6749 ... 0.645 0.9992
Attributes:
    model:     !KaiModel\n_dummy_scatterer: !Spheres\n  scatterers: [!Sphere ...
    strategy:  !EmceeStrategy\nnwalkers: 50\nnsamples: 1000\nnpixels: 8000\nw...
    time:      2280.9646060466766
    _kwargs:   {}\n
<xarray.DataArray (flat: 8000)>
array([0.95056503, 0.98485299, 1.00834467, ..., 0.91699669, 1.0232044 ,
       1.04089093])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 11.15 6.372 15.93 2.124

In [13]:
# seems like don't need model at least for current analysis
# end up with noise_sd as a coordinate which is weird
print(return_variable)
samples = return_variable.samples[:,999]
lnprob = return_variable.lnprobs[5]
# can get rid of noise_sd coord using .reset_coords('noise_sd', drop = True)
print(samples.reset_coords('noise_sd',drop=True))
print(lnprob.reset_coords('noise_sd',drop=True))
burnt_samples = return_variable.burn_in(110).samples[:,889]

SamplingResult(data=<xarray.DataArray (flat: 8000)>
array([0.93718599, 1.02949085, 1.1157527 , ..., 0.99884039, 1.0255741 ,
       0.95243352])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 52.21 65.14 55.4 58.41 ... 68.85 43.54 69.21 70.09
  * y        (flat) float64 108.1 96.46 105.7 125.7 ... 97.53 108.3 94.52 116.6
  * z        (flat) int64 0 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0 0
Attributes:
    _flat:               [[52.214999999999996, 108.14699999999999, 0], [65.13...
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
    noise_sd:            0.00862557822556247
    original_dims:       {'x': [40.71, 40.887, 41.064, 41.241, 41.418, 41.595..., model=<module 'holopy.inference.model' from '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holopy_code/holopy/holopy/inference/model.py'>, strategy=EmceeStrategy(nwalkers=50, nsamples=1000, npix

ValueError: One or more of the specified variables cannot be found in this dataset

In [15]:
# set results equal to loaded fit for further analysis
results5 = return_variable

# Data processing (ie drop pre-burn in data, drop non-converged fits, and decimate remaining data so its independent)

In [16]:
# set save path for figures (only necessary if loaded fit and not from earlier)
SAVEPATH = INITIALCONDPATH + LOAD_NAME_OF_FIT
print(SAVEPATH)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/on_top_of_each_other_geometry_with_noise/no_mod_initial_conditions_version_1/particle_swap_not_degenerate/von_Mises_Fisher_walkers_50_nsample_1000_fit_1


In [96]:
samples = results6.samples
print(samples[:,999][0])
print(means)

<xarray.DataArray (parameter: 11)>
array([1.58338314e+00, 6.61810399e-01, 5.84937320e+01, 1.08363985e-01,
       6.33215265e+00, 2.65374732e+00, 1.08652212e+02, 2.10359701e+01,
       1.59971242e+00, 6.53749957e-01, 6.43446763e-01])
Coordinates:
  * parameter  (parameter) <U5 'n_1' 'r_1' 'x_g' 'gap' ... 'n_2' 'r_2' 'alpha'
Attributes:
    acceptance_fraction:  0.4101799999999999
[1.583726196514745, 0.6664235732985598, 58.46192307204503, 0.11316391015815017, 6.3410849043146085, 2.660298049709442, 108.6570280816238, 21.026538216193, 1.6002167068691273, 0.649077996672116, 0.688996312318098]


In [97]:
print(samples[:,0].sel(parameter='phi'))

<xarray.DataArray (walker: 50)>
array([6.33720825, 6.33998815, 6.32349877, 6.34625714, 6.33353435,
       6.33913019, 6.33465505, 6.33372   , 6.33424197, 6.34160309,
       6.34008844, 6.32893577, 6.31841436, 6.33022361, 6.33423525,
       6.34057469, 6.33651475, 6.32733547, 6.33930499, 6.33368493,
       6.3382333 , 6.34473374, 6.34288955, 6.30700706, 6.33600083,
       6.33009705, 6.33390416, 6.33094803, 6.31835266, 6.337107  ,
       6.33199072, 6.32606402, 6.33986456, 6.32442276, 6.33453785,
       6.32500346, 6.32547301, 6.33129404, 6.33308461, 6.3386144 ,
       6.3231174 , 6.32333075, 6.33226935, 6.35098389, 6.31768577,
       6.32398468, 6.32664377, 6.33851072, 6.34253842, 6.34060844])
Coordinates:
    parameter  <U5 'phi'
Dimensions without coordinates: walker
Attributes:
    acceptance_fraction:  0.4101799999999999


In [41]:
print(initial_guess[0])

[1.58486443e+00 6.74989916e-01 5.85135853e+01 9.89276102e-02
 6.33477163e+00 2.65102329e+00 1.08650458e+02 2.10216152e+01
 1.60176563e+00 6.44553454e-01 6.20269077e-01]


In [98]:
# select gap parameter values for first walker (all chains)
gaps_example = samples[1].sel(parameter='gap')
print(gaps_example)

<xarray.DataArray (chain: 1000)>
array([0.11339539, 0.11317322, 0.11317322, 0.11317322, 0.11317322,
       0.11303293, 0.11298533, 0.11298421, 0.11306678, 0.11305209,
       0.11305209, 0.11303588, 0.1130036 , 0.1130036 , 0.1130036 ,
       0.1130036 , 0.11300245, 0.11300409, 0.11300399, 0.11300399,
       0.11300385, 0.11300385, 0.11300385, 0.11300385, 0.11308863,
       0.11307389, 0.11307389, 0.11306946, 0.11306946, 0.11306158,
       0.11306158, 0.11304786, 0.11304786, 0.11304786, 0.11297055,
       0.11294905, 0.11296492, 0.11296492, 0.11296492, 0.11296492,
       0.11293043, 0.11294824, 0.11294824, 0.11294824, 0.1128522 ,
       0.1128522 , 0.1128522 , 0.11282857, 0.11282857, 0.11282857,
       0.11282857, 0.11282857, 0.1127962 , 0.1127962 , 0.11278876,
       0.11263655, 0.11263655, 0.11263655, 0.11263655, 0.11263655,
       0.11263595, 0.11260948, 0.11269385, 0.11269385, 0.11264795,
       0.11264795, 0.11264795, 0.11264795, 0.11258174, 0.11258174,
       0.11254328, 0.11254328

In [99]:
plt.figure()
plt.plot(gaps_example)

### Drop pre-burn in (right now ad hoc but come back to make more systematic)

In [37]:
# plot pre-burn in
plt.figure()
plt.title('Pre burn-in logprob of fit')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
for i in range(11):
    plt.plot(results6.lnprobs[i])
plt.savefig(SAVEPATH + '/pre_burn_in_lnprob.png')

In [233]:
for i in range(4):
    #plt.plot(results5.lnprobs[i])
    print(results5.lnprobs[:,999][i])

<xarray.DataArray ()>
array(30694.85396699)
Attributes:
    acceptance_fraction:  0.3945333333333333
<xarray.DataArray ()>
array(30697.28410234)
Attributes:
    acceptance_fraction:  0.3945333333333333
<xarray.DataArray ()>
array(30701.46894588)
Attributes:
    acceptance_fraction:  0.3945333333333333
<xarray.DataArray ()>
array(30701.92375578)
Attributes:
    acceptance_fraction:  0.3945333333333333


In [39]:
# Use .burn_in() to chop off data before a specific sample number
cut_number = 100
burnt_results6 = results6.burn_in(cut_number) 
#120 seems good for 1000 somewhat random start
#300 for 2000 chain random start
#400 seems good for 3000 chain random start
plt.figure()
plt.title(f'Post burn-in logprob (cut off first {cut_number})')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
#ids = [1,3,8,9,10] (for 3000 chain)
for i in range(30):
    plt.plot(burnt_results6.lnprobs[i])
plt.savefig(SAVEPATH + '/many_lnprobs.png')
# to do more systematically could maybe calculate some sort of slope vs maximum slope cutoff?

In [38]:
# looking at the fits for all the different walkers it's clear that several don't converge well
plt.figure()
plt.title('Post burn-in logprob showing bad fits')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
for i in range(4):
    plt.plot(burnt_results5.lnprobs[i])
plt.savefig(SAVEPATH + '/few_lnprobs_to_show_bad_fits.png')

In [33]:
new_sample_length = len(burnt_results5.lnprobs[i])
print(burnt_results5.lnprobs[i][new_sample_length-1])

<xarray.DataArray ()>
array(30695.78594419)
Attributes:
    acceptance_fraction:  0.18866666666666668


### Visualize Data Traces

In [26]:
#samples = return_variable.samples
#print(samples.sel(parameter='theta'))

<xarray.DataArray 'samples' (walker: 50, chain: 1000)>
array([[2.652589, 2.652589, 2.654731, ..., 2.651085, 2.650997, 2.650997],
       [2.657331, 2.657331, 2.657374, ..., 2.649846, 2.650575, 2.650575],
       [2.634226, 2.634226, 2.637238, ..., 2.650072, 2.650232, 2.650232],
       ...,
       [2.666826, 2.666826, 2.652876, ..., 2.652025, 2.652025, 2.65243 ],
       [2.651763, 2.651265, 2.651265, ..., 2.651861, 2.651361, 2.651361],
       [2.650532, 2.650532, 2.651493, ..., 2.652114, 2.652114, 2.652114]])
Coordinates:
    parameter  <U5 'theta'
Dimensions without coordinates: walker, chain
Attributes:
    acceptance_fraction:  0.41628


In [27]:
#SAVEPATH = '/Volumes/manoharan_lab/kaitorrens/Analysis/Fits/depletion_holography/09-21-21/depletion/0.0875/03/test/mcmcdimer_frame0_mcmc_trace_plots'

In [29]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
#plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/few_theta_fits.png')

In [418]:
# plot theta mod pi
plt.figure()
plt.title('Theta fit mod pi')
plt.ylabel('Theta mod pi')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
plt.axhline(y=np.pi, color='gray', linestyle='-.', label='pi')
for i in range(30):
    theta = samples[i].sel(parameter='theta')
    for j in range(len(theta)):
        if theta[j] < -2:
            theta[j] = theta[j] + 2*np.pi
        elif theta[j] < 0.1:
            theta[j] = theta[j] + np.pi
    plt.plot(theta)
plt.savefig(SAVEPATH + '/many_theta_mod_pi_fits.png')

In [31]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
#plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/many_phi_fits.png')

In [38]:
# 18 starts at 2*pi for phi and 3 has phi off by pi
plt.figure()
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
plt.plot(samples[3].sel(parameter='gap'))

In [45]:
plt.figure()
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
plt.plot(samples[3].sel(parameter='phi'))

In [42]:
plt.figure()
plt.plot(burnt_results5.lnprobs[3])

In [695]:
# plot phi mod 2*pi
plt.figure()
plt.title('Phi fit mod 2pi')
plt.ylabel('Phi mod 2pi')
plt.xlabel('sample (chain)')
two_pi = 2*np.pi
plt.axhline(y=PHI+two_pi, color='gray', linestyle='--', label='real value')
swapped_index = []
for i in range(30):
    phi = results5.samples[i].sel(parameter='phi')
    for j in range(len(phi)):
        if phi[j] < 0.1:
            phi[j] = phi[j] + two_pi
    if all(phi > 4):
        plt.plot(phi)
    else:
        swapped_index.append(i)
plt.savefig(SAVEPATH + '/many_phi_mod_2*pi_fits.png')
print(swapped_index)

[]


In [33]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/few_gap_fits.png')

In [35]:
# look at some traces of r1
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/many_r1_fits.png')

In [41]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/few_r2_fits.png')

In [43]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
#plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/many_n1_fits.png')

In [45]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
#plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/few_n2_fits.png')

In [49]:
# look at some traces of x_g
plt.figure()
plt.title("Central x position (um) fit")
plt.ylabel('x (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Xg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(results5.samples[i].sel(parameter='x_g'))
plt.savefig(SAVEPATH + '/many_xg_fits.png')

In [51]:
# look at some traces of y_g
plt.figure()
plt.title("Central y position (um) fit")
plt.ylabel('y (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Yg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(results5.samples[i].sel(parameter='y_g'))
plt.savefig(SAVEPATH + '/few_yg_fits.png')

In [53]:
# look at some traces of z_g
plt.figure()
plt.title("Central z position (um) fit")
plt.ylabel('z (um)')
plt.xlabel('sample (chain)')
#plt.axhline(y=Zg_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(results5.samples[i].sel(parameter='z_g'))
plt.savefig(SAVEPATH + '/many_zg_fits.png')

In [96]:
print(samples.sel(parameter='n_2'))

<xarray.DataArray 'samples' (walker: 1000, chain: 50)>
array([[1.6028308 , 1.60276632, 1.60281288, ..., 1.60268985, 1.60284233,
        1.60277122],
       [1.6028308 , 1.60276632, 1.60281281, ..., 1.60268985, 1.60284233,
        1.60277122],
       [1.6028308 , 1.60276632, 1.60281281, ..., 1.60268985, 1.60284233,
        1.60277023],
       ...,
       [1.60679439, 1.60780986, 1.60676059, ..., 1.60808076, 1.60762128,
        1.60725447],
       [1.60678139, 1.60784512, 1.60676059, ..., 1.60808076, 1.60762128,
        1.60725447],
       [1.60677647, 1.60784512, 1.60676059, ..., 1.60808076, 1.60762128,
        1.60725447]])
Coordinates:
    parameter  <U3 'n_2'
Dimensions without coordinates: walker, chain
Attributes:
    acceptance_fraction:  0.40924


In [624]:
# look at some of the traces for walkers that don't converge
plt.plot(samples[7].sel(parameter='theta'))
plt.plot(samples[10].sel(parameter='theta'))

In [46]:
# look at theta that did converge and add pi to them
for i in range(4):
    plt.plot(samples[i].sel(parameter='theta')+np.pi)

In [54]:
plt.plot(samples[7].sel(parameter='phi'))
plt.plot(samples[10].sel(parameter='phi'))

In [57]:
plt.plot(samples[7].sel(parameter='gap'))
plt.plot(samples[10].sel(parameter='gap'))

### Drop bad convergence (ie low lnprob) walkers. Later attempt to fix these fits instead.

In [55]:
# look at distribution of final lnprob values across walkers
new_sample_length = len(burnt_results5.lnprobs[0])
plt.figure()
plt.plot(burnt_results5.lnprobs[:,(new_sample_length-1)])
plt.savefig(SAVEPATH + '/final_lnprob_values_across_walkers')
maxlnprob = max(burnt_results5.lnprobs[:,(new_sample_length-1)])
converged_value = maxlnprob - 0.02*maxlnprob
print(converged_value)

<xarray.DataArray 'lnprobs' ()>
array(13503.64377601)


In [57]:
# The bad fits are the swapped angles fits (visible in the reorganized pandas dataset)
# -> is there a good way to correct these or should I just drop them?
# Start by implementing a cutoff in lnprobs to drop them and then can work on fixing later
samples = burnt_results5.samples
converged_samples_nan = xr.DataArray()
# 0 doesn't work as a cutoff universally, example, 3000 chain fit 1 needs 15000 as cutoff
bad_fit_index = []
good_fit_index = []
for i in range(len(samples)):
    new_sample_length = len(burnt_results5.lnprobs[i])
    if burnt_results5.lnprobs[i][new_sample_length-1] > converged_value:
        converged_samples_nan = xr.concat([converged_samples_nan,samples[i]],'walker')
        # .append() isn't quite what we want, try to use xarray methods
        good_fit_index.append(i)
    else:
        bad_fit_index.append(i)
converged_samples = converged_samples_nan[1:]
print(converged_samples)
print(len(converged_samples))
print(bad_fit_index)

<xarray.DataArray (walker: 50, chain: 900, parameter: 11)>
array([[[ 1.58485129,  0.67479854, 58.47529414, ...,  1.60183538,
          0.64463213,  0.63968632],
        [ 1.58485129,  0.67479854, 58.47529414, ...,  1.60183538,
          0.64463213,  0.63968632],
        [ 1.58484587,  0.67481189, 58.47754795, ...,  1.60182196,
          0.64462673,  0.63896094],
        ...,
        [ 1.58471815,  0.66726066, 58.48229142, ...,  1.59935407,
          0.6476008 ,  0.64288574],
        [ 1.58476819,  0.66720039, 58.48226791, ...,  1.59936314,
          0.64756635,  0.64296229],
        [ 1.58476819,  0.66720039, 58.48226791, ...,  1.59936314,
          0.64756635,  0.64296229]],

       [[ 1.58482936,  0.67483351, 58.48004526, ...,  1.60181454,
          0.64461965,  0.63938897],
        [ 1.5848273 ,  0.67483158, 58.47954352, ...,  1.60181362,
          0.64461724,  0.6392534 ],
        [ 1.58483037,  0.67481833, 58.47966902, ...,  1.60181451,
          0.64463172,  0.63741142],
...
    

In [ ]:
# could also do this with ds.drop_sel(space=["IN", "IL"]) where we use walker = [drop indexes]

In [49]:
# plot converged samples
# need to modify this for each fit you want to use it for
# converged_id = [0,1,2,3,4,5,6,8,9]
plt.figure()
for id in good_fit_index:
    plt.plot(burnt_results5.lnprobs[id])

NameError: name 'converged_id' is not defined

#### Visualize good vs. bad fit parameter traces

In [437]:
# plot bad fits to see parrellels between them
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/30089_bad_theta_fits.png')

In [438]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/30089_bad_phi_fits.png')

In [439]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(results5.samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/30089_bad_gap_fits.png')

In [622]:
# plot good fits to see parrellels between them
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/30089_good_theta_fits.png')

In [ ]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/30089_good_phi_fits.png')

In [ ]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in good_fit_index:
    plt.plot(results5.samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/30089_good_gap_fits.png')

#### Visualize fits with low starting phi values (near boundary)

In [717]:
# look at how many starting phi look like the starting phi that lead to these bad fits
low_start_index = []
for i in range(50):
    if (samples[i,0].sel(parameter='phi') > np.pi) and (samples[i,0].sel(parameter='phi') < 2*np.pi):
        low_start_index.append(i)
print(low_start_index)

[0, 2, 5, 6, 9, 13, 18, 19, 20, 21, 22, 26, 27, 28, 38, 42, 44, 49]


In [613]:
# for this run 16 is bad since it swaps phi to around pi
low_start_index.remove(16)
print(low_start_index)

[0, 1, 2, 3, 5, 6, 7, 8, 9, 18, 22, 23, 25, 26, 27, 31, 36, 38, 40, 42, 43, 44, 45, 47]


In [718]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/low_starting_phi_lnprobs.png')

In [615]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id][100:])
plt.savefig(SAVEPATH + '/low_starting_phi_lnprobs_drop_first_220.png')

In [719]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/low_starting_phi_theta_fit.png')

In [720]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/low_starting_phi_phi_fit.png')

In [64]:
# look at some traces of phi that are never off by pi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    if all(samples[i,:].sel(parameter='phi') > 4):
        plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/low_starting_phi_phi_fit_not_off_by_pi.png')

In [121]:
# look at the end of some traces of phi that are not off by pi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    if all(samples[i,:].sel(parameter='phi')[-50:-1] > 4):
        plt.plot(samples[i].sel(parameter='phi')[-50:-1])

In [65]:
# plot starting phi positions for low initial conditions that aren't off by mod pi
plt.figure()
plot_varible = []
for id in low_start_index:
    if samples[id,0].sel(parameter='phi') > 4:
        plot_varible.append(samples[id,0].sel(parameter='phi'))
plt.plot(plot_varible)
plt.savefig(SAVEPATH + '/low_starting_phi_plot_of_starting_phi_not_off_by_pi.png')

In [721]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/low_starting_phi_gap_fit.png')

In [722]:
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/low_starting_phi_r1_fit.png')

In [723]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/low_starting_phi_r2_fit.png')

In [724]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/low_starting_phi_n1_fit.png')

In [725]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/low_starting_phi_n2_fit.png')

In [ ]:
# look at the fits that start with lnprob like the bad fits but then jump to good fits
# these are subset of low_start index and so already convered

#### Visualize fits with high starting phi values (near boundary)

In [726]:
# look at how many starting phi look like the starting phi that lead to bad fits
high_start_index = []
for i in range(50):
    if (samples[i,0].sel(parameter='phi') < 1):
        high_start_index.append(i)
print(high_start_index)

[30, 34, 35, 37, 41, 43, 45, 46, 48]


In [727]:
plt.figure()
for id in high_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/high_starting_phi_lnprobs.png')

In [728]:
plt.figure()
for id in high_start_index:
    plt.plot(burnt_results5.lnprobs[id][100:])
plt.savefig(SAVEPATH + '/high_starting_phi_lnprobs_drop_first_350.png')

In [729]:
# look at some traces of theta with high starting phi
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/high_starting_phi_theta_fit.png')

In [732]:
# look at some traces of phi with high starting phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/high_starting_phi_phi_fit.png')

In [731]:
# look at some traces of gap with high starting phi
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/high_starting_phi_gap_fit.png')

In [733]:
# look at some traces of r_1 with high starting phi
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/high_starting_phi_r1_fit.png')

In [734]:
# look at some traces of r2 with high starting phi
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/high_starting_phi_r2_fit.png')

In [735]:
# look at some traces of n1 with high starting phi
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/high_starting_phi_n1_fit.png')

In [736]:
# look at some traces of n2 with high starting phi
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in high_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/high_starting_phi_n2_fit.png')

#### Visualize fits with low starting theta values

In [442]:
# look at how many starting phi look like the starting phi that lead to these bad fits
low_start_index = []
for i in range(30):
    if (results5.samples[i,0].sel(parameter='theta') > np.pi) and (results5.samples[i,0].sel(parameter='phi') < 2*np.pi):
        low_start_index.append(i)
print(low_start_index)

[2, 5, 9, 12, 14, 15, 16, 19, 20, 21, 23, 24]


In [443]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/boundary_starting_theta_lnprobs.png')

In [446]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(results5.samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_theta_fit.png')

In [447]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_phi_fit.png')

In [449]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_gap_fit.png')

In [450]:
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_r1_fit.png')

In [451]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_r2_fit.png')

In [452]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_n1_fit.png')

In [453]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/boundary_starting_theta_n2_fit.png')

### Decimate data so just keep independent fits

In [58]:
# look at autocorrelation to see how long it is before lose memory so can decimate into independent samples
series = pd.Series(converged_samples[1].sel(parameter='gap'))
autocorr_as_function_of_time = []
for i in range(len(series)):
    autocorr = series.autocorr(lag=i)
    autocorr_as_function_of_time.append(autocorr)
plt.figure()
plt.plot(autocorr_as_function_of_time)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


In [167]:
# look at what corresponding trace looks like
plt.figure()
plt.plot(converged_samples[1].sel(parameter='gap'))

In [60]:
# look at autocorrelation of gap data from different walkers
plt.figure()
plt.title('autocorrelation of gap')
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='gap'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)
plt.savefig(SAVEPATH + '/auto_correlation_of_gap.png')

In [221]:
# look at trace of gap from different walkers
plt.figure()
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='gap'))

In [130]:
# look at autocorrelation of theta data from different walkers
plt.figure()
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='theta'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)

In [171]:
# look at trace of theta from different walkers
plt.figure
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='theta'))

In [185]:
# look at trace of n_1 from different walkers
plt.figure
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='n_1'))

In [103]:
# We can use the following notation to cycle throught the different parameter labels
for parameter in converged_samples.coords['parameter'].data:
    print(parameter)

n_1
r_1
x_g
gap
phi
theta
y_g
z_g
n_2
r_2
alpha


In [61]:
# Plot autocorrelation for all the different parameters
# set up plotting
number_columns = int(np.ceil(len(converged_samples.coords['parameter'].data)/3))
fig,axes = plt.subplots(3,number_columns)
m = 1
# go through analysis
for parameter_name in converged_samples.coords['parameter'].data:
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
        plt.subplot(3,number_columns,m)
        plt.plot(autocorr_as_function_of_time)
    # label plot
    row = int(np.floor((m-1)/4))
    column = int(m-4*row)
    axes[row,column-1].set_title(parameter_name)
    fig.supxlabel('Chain Number')
    fig.supylabel('Autocorrelation')
    # increment number tracker
    m = m + 1
plt.savefig(SAVEPATH + '/all_autocorrelation.png')

In [62]:
# find the correlation time for each of these parameters (ie when autocorrelation drops to 0 for the first time)
all_parameter_indices = []
for parameter_name in converged_samples.coords['parameter'].data:
    all_walker_indices = []
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
            if autocorr < 0:
                index = i
                all_walker_indices.append(index)
                break
            elif i == (len(series)-1):
                index = i
                all_walker_indices.append(index)
                print("sample " + str(n) + " of " +  parameter_name + " remains correlated")
    all_parameter_indices.append(all_walker_indices)
print(all_parameter_indices)
# Q: should I find max correlation time or mean correlation time for each parameter?
# start with easiest which is just overall max
overall_correlation_time = np.max(all_parameter_indices)
print(overall_correlation_time)

[[186, 169, 170, 158, 155, 174, 176, 187, 179, 182, 172, 196, 170, 148, 201, 306, 194, 178, 236, 181, 171, 186, 194, 181, 188, 179, 172, 173, 215, 211, 187, 178, 162, 172, 176, 171, 265, 179, 185, 174, 189, 190, 182, 206, 200, 185, 186, 216, 199, 204], [496, 475, 578, 551, 624, 590, 585, 515, 619, 723, 486, 821, 700, 855, 653, 590, 648, 619, 760, 744, 679, 535, 831, 841, 658, 686, 555, 519, 558, 855, 662, 852, 818, 496, 770, 879, 571, 788, 705, 549, 486, 514, 501, 752, 493, 663, 544, 571, 800, 546], [399, 442, 397, 625, 376, 593, 482, 801, 603, 503, 448, 624, 588, 593, 534, 599, 438, 493, 462, 475, 387, 605, 671, 516, 463, 733, 135, 381, 565, 477, 588, 623, 435, 410, 467, 504, 583, 440, 501, 492, 566, 512, 352, 483, 462, 520, 383, 498, 463, 410], [112, 155, 122, 135, 123, 83, 141, 121, 143, 132, 141, 157, 206, 157, 129, 174, 200, 136, 132, 143, 144, 112, 120, 136, 134, 117, 122, 129, 150, 127, 149, 111, 141, 145, 156, 142, 162, 155, 130, 143, 116, 142, 106, 130, 168, 159, 123, 159, 117

In [63]:
# other approach where we find the mean and then take the max
mean_parameter_corr_time = np.mean(all_parameter_indices, axis=1)
max_of_mean_parameter_corr_time = int(np.max(mean_parameter_corr_time))
print(max_of_mean_parameter_corr_time)

646


In [64]:
# now use correlation time to decimate the data
chain_number = len(converged_samples[0])
number_ind_samples_per_walker = int(np.ceil(chain_number/overall_correlation_time))
independent_samples =[]
for i in range(number_ind_samples_per_walker):
    index = (chain_number-1-overall_correlation_time*i)
    if i==0:
        independent_samples = converged_samples[:,index]
    else:
        independent_samples = xr.concat([independent_samples,(converged_samples[:,index])], 'walker')

In [65]:
# now convert independent samples into a form that can be input into seaborn pairplot
independent_samples_pd = independent_samples.to_dataframe(name = 'independent samples')
independent_samples_pd = independent_samples_pd.reset_index()
independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')
#print(independent_samples_pd)

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_20804/3190556392.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')


In [67]:
full_pair_plot = sns.pairplot(independent_samples_pd)
full_pair_plot.savefig(SAVEPATH + '/pair_plot_of_all_samples')

In [68]:
# seems like the parameter sets from the end of the run and the beginning of the run
# cluster in different ways, lets try seperating them

# get parameters at the end of the fitting process
end_of_run_ind = independent_samples[:len(converged_samples)]
end_of_run_ind_pd = end_of_run_ind.to_dataframe(name = 'independent samples')
end_of_run_ind_pd = end_of_run_ind_pd.reset_index()
end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
print(end_of_run_ind_pd)

# get parameters from earlier on in the fitting process (~correlation time before the end)
early_run_ind = independent_samples[len(converged_samples):]
early_run_ind_pd = early_run_ind.to_dataframe(name = 'independent samples')
early_run_ind_pd = early_run_ind_pd.reset_index()
early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')
#print(early_run_ind_pd)

parameter     alpha       gap       n_1       n_2       phi       r_1  \
walker                                                                  
0          0.642962  0.116010  1.584768  1.599363  0.050827  0.667200   
1          0.640514  0.114075  1.584340  1.599278  0.053252  0.668308   
2          0.639627  0.112077  1.584613  1.599251  0.053311  0.667528   
3          0.640743  0.115330  1.584367  1.598718  0.053157  0.667653   
4          0.641095  0.115408  1.584844  1.599047  0.055316  0.667586   
5          0.643222  0.118253  1.584545  1.598913  0.049810  0.667997   
6          0.642946  0.113724  1.584870  1.598906  0.053086  0.667892   
7          0.640990  0.116527  1.584396  1.599609  0.052980  0.668641   
8          0.640224  0.115372  1.584320  1.599078  0.051470  0.667787   
9          0.640252  0.112594  1.584496  1.599911  0.050850  0.667853   
10         0.643280  0.111409  1.584062  1.599549  0.049602  0.668232   
11         0.640397  0.113937  1.584744  1.599375  

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_20804/2063201981.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_20804/2063201981.py:15: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')


In [69]:
end_of_run_pair_plot = sns.pairplot(end_of_run_ind_pd)
end_of_run_pair_plot.savefig(SAVEPATH + '/pair_plot_of_end_of_run_samples')
# hmm seems like one of the fits is comparatively bad and is an outlier (for 1000 chain, random starting)

In [70]:
early_run_pair_plot = sns.pairplot(early_run_ind_pd)
early_run_pair_plot.savefig(SAVEPATH + '/pair_plot_of_early_run_samples')

In [142]:
# return real fit values
starting_means = []
for p in model6._parameters:
        starting_means.append(p.mu)
print(starting_means)

[1.5837615975996668, 0.666459947416345, 58.484832600004395, 0.11289545130974342, 6.3318712339240255, 2.6508866488913227, 108.65291035266634, 21.054742080951772, 1.6001721492809295, 0.6491023121734101, 0.6465348913785244]


In [144]:
# compare average of end points of converged fits with ground truth values
mean_prediction = np.mean(end_of_run_ind_pd,axis=0)
print(mean_prediction)
print(starting_means)
mean_differences = np.zeros(len(starting_means))
corresponding_id = [2,5,8,1,4,7,9,10,3,6,0]
for i in range(len(starting_means)):
    mean_differences[i] = starting_means[i]-mean_prediction[corresponding_id[i]]
print(mean_differences)

parameter
alpha      0.644564
gap        0.109737
n_1        1.583458
n_2        1.599162
phi        6.330971
r_1        0.661251
r_2        0.654057
theta      2.654173
x_g       58.493734
y_g      108.651966
z_g       21.030329
dtype: float64
[1.5837615975996668, 0.666459947416345, 58.484832600004395, 0.11289545130974342, 6.3318712339240255, 2.6508866488913227, 108.65291035266634, 21.054742080951772, 1.6001721492809295, 0.6491023121734101, 0.6465348913785244]
[ 0.00030376  0.0052094  -0.00890111  0.00315829  0.00090066 -0.00328601
  0.00094411  0.02441259  0.00101032 -0.00495494  0.00197072]


In [145]:
mean_prediction = np.mean(end_of_run_ind_pd,axis=0)
mean_differences = mean_prediction.copy()
opp_corresponding_id = [10,3,0,8,4,1,9,5,2,6,7]
for i in range(len(starting_means)):
    mean_differences[i] = starting_means[opp_corresponding_id[i]]-mean_prediction[i]
print(mean_prediction)
print(mean_differences)

parameter
alpha      0.644564
gap        0.109737
n_1        1.583458
n_2        1.599162
phi        6.330971
r_1        0.661251
r_2        0.654057
theta      2.654173
x_g       58.493734
y_g      108.651966
z_g       21.030329
dtype: float64
parameter
alpha    0.001971
gap      0.003158
n_1      0.000304
n_2      0.001010
phi      0.000901
r_1      0.005209
r_2     -0.004955
theta   -0.003286
x_g     -0.008901
y_g      0.000944
z_g      0.024413
dtype: float64


In [86]:
type(mean_differences)

pandas.core.series.Series

In [146]:
mean_differences.to_csv(SAVEPATH+'/13627_cutoff_initial_values_minus_mean_of_fits')

In [471]:
# test is cos(phi-previous_phi) could be leading to problems in lnprior
phi = [0,2*np.pi,2*np.pi,4*np.pi,4*np.pi,0]
previous_phi = [np.pi/4,np.pi/4,9*np.pi/4,np.pi/4,9*np.pi/4,-7*np.pi/4]
for i in range(len(phi)):
    print(np.cos(phi[i]-previous_phi[i]))
# looks like it works totally fine

0.7071067811865476
0.7071067811865474
0.7071067811865476
0.7071067811865466
0.7071067811865474
0.7071067811865474
